In [1]:
import pandas as pd

fspd_f = pd.read_stata('./data/March30_f.dta')

In [2]:
fspd_f.columns

Index(['income_group', 'region_wb', 'fsd_group', 'country', 'segment', 'lever',
       'policydecision_details', 'db', 'level', 'covid_mentioned',
       'policy_description', 'iso', 'initialdate', 'FAPDA_region',
       'FAPDA_subregion', 'policydirection', 'policyphase',
       'contextoradditionalinfo', 'decision_making_institution', 'budget',
       'beneficiaries', 'commodity1', 'commodity2', 'commodity3',
       'additionalcommodities', 'term', 'targeted', 'emergency',
       'foodsecuritydimension', 'source1name', 'source1link', 'source2name',
       'source2link', 'source3name', 'source3link', 'url', 'topics', 'source',
       'yearoflastrevision', 'revisiondetails', 'generaltaxbase',
       'specifictaxbase', 'rate', 'exemptiontype', 'exemptiondetails',
       'earmarking', 'policy_part', 'policycategorycode', 'gendermainstreamed',
       'nutritionmainstreamed', 'framework', 'count', 'year', 'policy_code',
       'id', 'y_start', 'y_end', 'inclusion', 'envitarget', 'diethealt

In [3]:
print(fspd_f['segment'].unique())
print(fspd_f['segment'].isnull().sum())

['04Trade' '02Farming' '11General_framework' '01Inputs'
 '10Natural_resource_mgmt' '03Distribution_transport'
 '08Final_consumption' '09Multiple' '05Processing_manufacturing'
 '06Retail' '07Foodservice']
0


In [4]:
# create a new dataframe with only 'policydecision_details', 'segment', and 'lever' columns

df = fspd_f[['policydecision_details', 'segment']]

In [5]:
# check minimum and maximum length of text in 'policydecision_details' column

print(df['policydecision_details'].str.len().min())
print(df['policydecision_details'].str.len().max())

3
7234


In [6]:
# check how many rows have len(policydecision_details) < 20

df[df['policydecision_details'].str.len() < 10].shape

(257, 2)

In [7]:
# encode all segments to numeric by keeping only the first two characters

df['segment'] = df['segment'].str[:2]

# then print the unique values of the 'segment' column

print(df['segment'].unique())

# finally, print the number of rows with null values in the 'segment' column (just to be sure)

print(df['segment'].isnull().sum())

# and now convert the 'segment' column to numeric

df['segment'] = pd.to_numeric(df['segment'])

['04' '02' '11' '01' '10' '03' '08' '09' '05' '06' '07']
0


/tmp/ipykernel_19380/2732274441.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['segment'] = df['segment'].str[:2]
/tmp/ipykernel_19380/2732274441.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['segment'] = pd.to_numeric(df['segment'])


In [ ]:
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

# remove stopwords from the 'policydecision_details' column

df['policydecision_details'] = df['policydecision_details'].apply(lambda x: ' '.join([word for word in x.split() if word not in (STOPWORDS)]))

# remove punctuation from the 'policydecision_details' column

df['policydecision_details'] = df['policydecision_details'].str.replace('[^\w\s]','')

In [8]:
import tensorflow as tf
from tensorflow.keras import models, layers
import tensorflow_datasets as tfds
import tensorflow_hub as hub
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.corpus import stopwords
import numpy as np

seed = 777

# randomly divide the data into train and validation sets (80% and 20% respectively)

train_df = df.sample(frac=0.8, random_state=seed)
val_df = df.drop(train_df.index)

# transform the data into tf.data.Dataset objects

train_ds_orig = tf.data.Dataset.from_tensor_slices((train_df['policydecision_details'].values, train_df['segment'].values))
train_ds = train_ds_orig.cache().shuffle(777).batch(64)
val_ds_orig = tf.data.Dataset.from_tensor_slices((val_df['policydecision_details'].values, val_df['segment'].values))
val_ds = val_ds_orig.cache().shuffle(777).batch(64)

2022-11-30 03:20:36.086774: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-11-30 03:20:37.458818: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/nico/anaconda3/envs/nicolandia/lib
2022-11-30 03:20:37.458904: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/nico/anaconda3/envs/nicolandia/lib
2022-11-30 03:20:37.458913: W tensorflow/compile

In [9]:
def render_history(history):
    plt.title("Losses")
    plt.plot(history.history["loss"], label="loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.legend()
    plt.show()
    plt.close()

    plt.title("Accuracies")
    plt.plot(history.history["accuracy"], label="accuracy")
    plt.plot(history.history["val_accuracy"], label="val_accuracy")
    plt.legend()
    plt.show()
    plt.close()

In [10]:
vocabulary_size = 10_000
sequence_length = 256

encoder = layers.TextVectorization(
    max_tokens=vocabulary_size,
    output_sequence_length=sequence_length,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    pad_to_max_tokens=True
)
encoder.adapt(train_ds_orig.map(lambda text, label: text).batch(256))

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


In [11]:
# This model has 3 LSTM layers stacked

model = models.Sequential()
model.add(encoder)
model.add(layers.Embedding(input_dim=vocabulary_size, output_dim=32))
model.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
model.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
model.add(layers.Bidirectional(layers.LSTM(64)))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(12, activation='softmax'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_vectorization (TextVec  (None, 128)              0         
 torization)                                                     
                                                                 
 embedding (Embedding)       (None, 128, 32)           320000    
                                                                 
 bidirectional (Bidirectiona  (None, 128, 128)         49664     
 l)                                                              
                                                                 
 bidirectional_1 (Bidirectio  (None, 128, 128)         98816     
 nal)                                                            
                                                                 
 bidirectional_2 (Bidirectio  (None, 128)              98816     
 nal)                                                   

In [12]:
model.compile(
    optimizer='adam', #<- one way
    #optimizer=tf.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# instead of adam, use one-cycle learning rate

lr_schedule = tf.keras.callbacks.LearningRateScheduler(
    lambda epoch: 1e-8 * 10**(epoch / 20))
history = model.fit(train_ds, epochs=5, validation_data=val_ds, callbacks=[lr_schedule])

Epoch 1/5


: 

: 

In [13]:
history = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
)

# save a checkpoint of the model every 5 epochs

checkpoint_path = "./checkpoints/cp-{epoch:04d}.ckpt"

checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model's weights every 5 epochs

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    verbose=1,
    save_weights_only=True,
    period=5)

# Train the model with the new callback

history = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=[cp_callback]
)

Epoch 1/10


: 

: 

In [ ]:
render_history(history)